## Filtering Down Coal Plant Data

### Load from raw excel data

In [1]:
import pandas as pd

# Load the Units sheet from the Global Coal Plant Tracker
excel_path = '../01_raw_data/coal_plant_data/Global-Coal-Plant-Tracker-October-2025-Supplement-Proposals-outside-of-China.xlsx'
df = pd.read_excel(excel_path, sheet_name='Units', header=0)

print(f"Loaded {len(df)} total records")

# Filter to US plants only
df_us = df[df['Country/Area'] == 'United States']
print(f"US plants: {len(df_us)}")

# Filter out retired plants (keep only rows where 'Retired Year' is null/NaN)
df_us = df_us[df_us['Retired year'].isna()]
print(f"After removing retired plants: {len(df_us)}")

# Keep only operating plants
df_us = df_us[df_us['Status'] == 'operating']
print(f"After filtering to 'operating' status: {len(df_us)}")

# Filter out waste coal plants
df_us = df_us[df_us['Coal type'] != 'waste coal']
print(f"After removing waste coal plants: {len(df_us)}")

# Deduplicate by GEM location ID — the Units sheet has one row per unit,
# so multi-unit plants appear multiple times at the same coordinates.
# Keep the unit with the highest capacity to represent each plant location.
df_us = df_us.sort_values('Capacity (MW)', ascending=False).drop_duplicates(subset=['GEM location ID']).sort_index()
print(f"After deduplicating by location: {len(df_us)}")

# preview the first few rows of the filtered dataframe
print(df_us.head())

Loaded 14363 total records
US plants: 1221
After removing retired plants: 439
After filtering to 'operating' status: 391
After removing waste coal plants: 377
After deduplicating by location: 181
        Database GEM unit/phase ID GEM location ID   Country/Area  \
13082  July 2025     G100000100057   L100000104168  United States   
13094  July 2025     G100000100130   L100000103996  United States   
13102  July 2025     G100000100165   L100000104249  United States   
13104  July 2025     G100000100277   L100000104089  United States   
13106  July 2025     G100000100289   L100000103787  United States   

                                                Wiki URL  \
13082  https://www.gem.wiki/AES_Puerto_Rico_power_sta...   
13094  https://www.gem.wiki/Allen_S._King_Generating_...   
13102                    https://www.gem.wiki/Amos_Plant   
13104       https://www.gem.wiki/Antelope_Valley_Station   
13106     https://www.gem.wiki/Apache_Generating_Station   

                           P

In [2]:
# Save to processed data folder
output_path = '../02_processed_data/active_coal_plants_US.csv'
df_us.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to ../02_processed_data/active_coal_plants_US.csv
